# Tour Ceylon — Component 4: Model Training (Live)
**R26-IT-152 · Ayurveda & Spiritual Tourism Matching System · D.C.M. De Silva (IT22108586)**

This notebook trains **every ML model used by the system from raw data**, prints each model's
parameters, computes all evaluation metrics live, and saves the exact `.pkl` files the Flask app loads.

| # | Model | Algorithm | Files produced |
|---|---|---|---|
| 1 | Dosha Classifier | Random Forest (selected from 6 algorithms) | `model_comparison_results.csv` |
| 2 | Review Safety Classifier | TF-IDF + linear SVM | `safety_classifier_model.pkl` |
| 3 | Symptom Classifier × 5 languages | TF-IDF + linear SVM | `symptom_classifier_model*.pkl` |
| 4 | Spiritual Retreat Classifier × 5 languages | TF-IDF + linear SVM | `spiritual_classifier_model_*.pkl` |
| 5 | Treatment Outcome Predictor | Random Forest Regressor | `outcome_predictor_model.pkl` |

**Run cells top to bottom (Shift+Enter).**

## 0. Setup — install libraries and upload data

In [ ]:
# scikit-learn is pinned to the same version the Flask app uses,
# so the saved .pkl files load without errors in the backend.
!pip install -q scikit-learn==1.8.0 openpyxl
import sklearn, pandas as pd, numpy as np
print("scikit-learn:", sklearn.__version__)

In [ ]:
# Upload training_data.zip (contains the data/ folder)
import os, zipfile
try:
    from google.colab import files
    uploaded = files.upload()
    zip_name = next(iter(uploaded))
    with zipfile.ZipFile(zip_name) as z:
        z.extractall(".")
except ImportError:
    print("Not running in Colab - using local data/ folder")

os.makedirs("models", exist_ok=True)
print("Data files:", sorted(os.listdir("data")))

## 1. Dosha Classifier — Random Forest
Predicts the tourist's primary dosha (Vata / Pitta / Kapha) from 6 profile features.
### 1.1 Load the dataset

In [ ]:
df = pd.read_excel("data/tourist_profiles_training_data.xlsx")
print("Total profiles:", len(df))
print("\nData origin:")
print(df["Data Origin"].value_counts().to_string())
print("\nDosha distribution:")
print(df["Primary Dosha (Vikruti)"].value_counts().to_string())
df.head()

### 1.2 Preprocess
- `Balanced (N/A)` profiles are removed (no single dominant dosha to learn).
- Text features are converted to numbers with **LabelEncoder**.
- 80 / 20 **stratified** train/test split (keeps the same dosha ratio in both sets).

In [ ]:
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split

TARGET = "Primary Dosha (Vikruti)"
FEATURES = ["Age", "Gender", "Sleep Pattern", "Digestion/Appetite",
            "Cold/Heat Sensitivity", "Emotional Tendency"]

d = df[df[TARGET] != "Balanced (N/A)"].copy()
X = d[FEATURES].copy()
for col in FEATURES[1:]:
    X[col] = LabelEncoder().fit_transform(X[col].astype(str))
y = d[TARGET]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y)
print(f"Used for training: {len(d)} profiles  ->  train {len(X_train)} / test {len(X_test)}")

### 1.3 Compare 6 algorithms (same split, same data)

In [ ]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.neural_network import MLPClassifier
from sklearn.naive_bayes import GaussianNB
from sklearn.svm import SVC
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

candidates = {
    "Random Forest": RandomForestClassifier(n_estimators=100, random_state=42),
    "Decision Tree": DecisionTreeClassifier(random_state=42),
    "Neural Network (ANN)": MLPClassifier(max_iter=1000, random_state=42),
    "Naive Bayes": GaussianNB(),
    "SVM": SVC(random_state=42),
    "KNN": KNeighborsClassifier(),
}
rows = []
for name, m in candidates.items():
    m.fit(X_train, y_train)
    p = m.predict(X_test)
    rows.append({"Model": name,
                 "Accuracy": round(accuracy_score(y_test, p), 3),
                 "Precision": round(precision_score(y_test, p, average="weighted", zero_division=0), 3),
                 "Recall": round(recall_score(y_test, p, average="weighted", zero_division=0), 3),
                 "F1-Score": round(f1_score(y_test, p, average="weighted", zero_division=0), 3)})
comparison = pd.DataFrame(rows).sort_values("Accuracy", ascending=False)
comparison.to_csv("models/model_comparison_results.csv", index=False)
comparison

### 1.4 Selected model — Random Forest: parameters

In [ ]:
rf = RandomForestClassifier(n_estimators=100, random_state=42)
rf.fit(X_train, y_train)
for k, v in rf.get_params().items():
    print(f"{k:28} {v}")

### 1.5 Evaluation — confusion matrix, per-class report, 5-fold cross-validation

In [ ]:
from sklearn.metrics import classification_report, confusion_matrix, ConfusionMatrixDisplay
from sklearn.model_selection import cross_val_score, StratifiedKFold
import matplotlib.pyplot as plt

pred = rf.predict(X_test)
print(classification_report(y_test, pred, zero_division=0))

cv = cross_val_score(RandomForestClassifier(n_estimators=100, random_state=42), X, y,
                     cv=StratifiedKFold(5, shuffle=True, random_state=42))
print("5-fold CV accuracy per fold:", [f"{s*100:.1f}%" for s in cv])
print(f"5-fold CV mean: {cv.mean()*100:.1f}% ± {cv.std()*100:.1f}%")

ConfusionMatrixDisplay(confusion_matrix(y_test, pred, labels=rf.classes_),
                       display_labels=rf.classes_).plot(cmap="Blues")
plt.title("Dosha Classifier — Confusion Matrix (test set)")
plt.show()

### 1.6 Feature importance — which inputs drive the prediction

In [ ]:
imp = pd.Series(rf.feature_importances_, index=FEATURES).sort_values()
imp.plot.barh(title="Random Forest feature importance")
plt.xlabel("Importance"); plt.show()
print(imp.sort_values(ascending=False).round(3).to_string())

> The Flask app (`matching_engine.py`) trains this **same Random Forest with the same parameters and
> features** on startup, so the app's predictions come from this model — not from fixed values.

## 2. Review Safety Classifier — TF-IDF + linear SVM
Flags reviews that mention hygiene, safety, harassment or scam issues.
Evaluated with **Leave-One-Out CV** (every review is tested once by a model that never saw it),
because the labelled dataset is small.

In [ ]:
import json, pickle
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.model_selection import LeaveOneOut, cross_val_predict

raw = json.load(open("data/labeled_reviews.json", encoding="utf-8"))
texts = raw["concerning"] + raw["not_concerning"]
labels = np.array([1]*len(raw["concerning"]) + [0]*len(raw["not_concerning"]))
print(f"{len(texts)} real reviews: {labels.sum()} concerning, {(labels==0).sum()} not concerning")

vec = TfidfVectorizer(ngram_range=(1, 2), stop_words="english", max_features=500)
Xs = vec.fit_transform(texts)
svm = SVC(kernel="linear", probability=True, class_weight="balanced", random_state=42)

print("\nTF-IDF parameters:", {k: vec.get_params()[k] for k in ["ngram_range", "stop_words", "max_features"]})
print("SVM parameters:   ", {k: svm.get_params()[k] for k in ["kernel", "C", "class_weight", "probability"]})
print("Vocabulary size:   ", len(vec.vocabulary_))

p = cross_val_predict(svm, Xs, labels, cv=LeaveOneOut())
safety_metrics = {"accuracy": accuracy_score(labels, p), "precision": precision_score(labels, p, zero_division=0),
                  "recall": recall_score(labels, p, zero_division=0), "f1": f1_score(labels, p, zero_division=0)}
print("\nLOO-CV:", {k: round(v, 3) for k, v in safety_metrics.items()})
print(confusion_matrix(labels, p))

svm.fit(Xs, labels)
pickle.dump({"vectorizer": vec, "model": svm, "model_name": "SVM",
             "n_examples": len(texts), "loo_metrics": safety_metrics},
            open("models/safety_classifier_model.pkl", "wb"))
print("\nSaved models/safety_classifier_model.pkl")

t = "I cannot recommend it highly enough, the staff were wonderful"
print(f"Test: '{t}' ->", "CONCERNING" if svm.predict(vec.transform([t]))[0] else "not concerning")

## 3 & 4. Free-text Classifiers — 5 languages × 2 domains (10 models)
One **separate model per language** (EN, DE, RU, ZH, JA), trained on native-language examples.
- Word n-grams for EN / DE / RU; **character n-grams** for ZH / JA (no spaces between words).
- English uses English stop-word removal.

In [ ]:
LANGS = ["en", "de", "ru", "zh", "ja"]

def train_text_model(data_file, out_file):
    data = json.load(open(data_file, encoding="utf-8"))
    X_txt = [t for t, c in data]; y_txt = np.array([c for t, c in data])
    lang = out_file.split("_")[-1].replace(".pkl", "")
    analyzer = "char_wb" if lang in ("zh", "ja") else "word"
    v = TfidfVectorizer(analyzer=analyzer, ngram_range=(1, 2), max_features=500,
                        stop_words="english" if lang in ("en", "model") else None)
    Xv = v.fit_transform(X_txt)
    m = SVC(kernel="linear", probability=True, random_state=42)
    loo_acc = accuracy_score(y_txt, cross_val_predict(m, Xv, y_txt, cv=LeaveOneOut()))
    m.fit(Xv, y_txt)
    pickle.dump({"vectorizer": v, "model": m, "model_name": "SVM",
                 "n_examples": len(X_txt), "loo_accuracy": loo_acc}, open(out_file, "wb"))
    return {"file": os.path.basename(out_file), "analyzer": analyzer, "examples": len(X_txt),
            "classes": len(set(y_txt)), "vocab": len(v.vocabulary_), "LOO accuracy": f"{loo_acc*100:.1f}%"}

In [ ]:
# 3. Ayurveda symptom classifiers (12 conditions)
sym = []
for lang in LANGS:
    src = "data/symptom_training_data_en.json" if lang == "en" else f"data/symptom_training_data_{lang}.json"
    out = "models/symptom_classifier_model.pkl" if lang == "en" else f"models/symptom_classifier_model_{lang}.pkl"
    r = train_text_model(src, out); r["lang"] = lang; sym.append(r)
pd.DataFrame(sym)[["lang", "file", "analyzer", "examples", "classes", "vocab", "LOO accuracy"]]

In [ ]:
# 4. Spiritual retreat classifiers (6 retreat types)
spi = []
for lang in LANGS:
    r = train_text_model(f"data/spiritual_training_data_{lang}.json",
                         f"models/spiritual_classifier_model_{lang}.pkl")
    r["lang"] = lang; spi.append(r)
pd.DataFrame(spi)[["lang", "file", "analyzer", "examples", "classes", "vocab", "LOO accuracy"]]

## 5. Treatment Outcome Predictor — Random Forest Regressor
Predicts expected improvement (%) from age, dosha, condition and treatment duration.
Trained on **6 real domain-expert patient cases + 48 expert-informed variations**; evaluated with LOO-CV.

In [ ]:
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, r2_score

o = pd.DataFrame(json.load(open("data/outcome_training_data.json")))
print(o["source"].value_counts().to_string(), "\n")

le_d = LabelEncoder().fit(o["dosha"]); le_c = LabelEncoder().fit(o["condition"])
Xo = pd.DataFrame({"age": o["age"], "dosha_enc": le_d.transform(o["dosha"]),
                   "condition_enc": le_c.transform(o["condition"]), "duration_days": o["duration_days"]})
yo = o["outcome_pct"]

rfr = RandomForestRegressor(n_estimators=100, random_state=42)
print("Parameters:", {k: rfr.get_params()[k] for k in ["n_estimators", "max_depth", "min_samples_leaf", "random_state"]})

loo_pred = cross_val_predict(rfr, Xo, yo, cv=LeaveOneOut())
mae, r2 = mean_absolute_error(yo, loo_pred), r2_score(yo, loo_pred)
print(f"LOO-CV  MAE = {mae:.1f} percentage points   R² = {r2:.3f}")

rfr.fit(Xo, yo)
pickle.dump({"model": rfr, "dosha_encoder": le_d, "condition_encoder": le_c,
             "n_real_cases": int((o["source"] == "real_case").sum()),
             "n_augmented": int((o["source"] != "real_case").sum()),
             "loo_mae": float(mae), "loo_r2": float(r2)},
            open("models/outcome_predictor_model.pkl", "wb"))
print("Saved models/outcome_predictor_model.pkl")

plt.scatter(yo, loo_pred); plt.plot([yo.min(), yo.max()], [yo.min(), yo.max()], "r--")
plt.xlabel("Actual outcome %"); plt.ylabel("Predicted (LOO)"); plt.title("Outcome Predictor — actual vs predicted")
plt.show()

## 6. Summary + download the trained models

In [ ]:
summary = {
    "dosha_rf": {"test_accuracy": float(comparison.iloc[0]["Accuracy"]), "test_f1": float(comparison.iloc[0]["F1-Score"]),
                 "cv5_mean": float(cv.mean()), "cv5_std": float(cv.std())},
    "safety_svm": {k: round(v, 3) for k, v in safety_metrics.items()},
    "symptom_svm_loo": {r["lang"]: r["LOO accuracy"] for r in sym},
    "spiritual_svm_loo": {r["lang"]: r["LOO accuracy"] for r in spi},
    "outcome_rfr": {"loo_mae": round(mae, 2), "loo_r2": round(r2, 3)},
    "sklearn_version": sklearn.__version__,
}
json.dump(summary, open("models/training_metrics.json", "w"), indent=2)
print(json.dumps(summary, indent=2))

import shutil
shutil.make_archive("trained_models", "zip", "models")
print("\nFiles:", sorted(os.listdir("models")))
try:
    from google.colab import files
    files.download("trained_models.zip")
except ImportError:
    pass

**After download:** copy every `.pkl` into `backend/` and `model_comparison_results.csv` into `backend/data/`,
then restart `python app.py`.